# Autonomous Agent Prediction (Beta) - Starter & AutoML Guide

Welcome to the **Autonomous Agent Prediction (Beta)** competition! In this competition, instead of submitting predictions directly, you submit an **autonomous machine learning agent** that is evaluated offline in a offline sandboxed environment. Your agent will dynamically load datasets, train models, select best submissions, and run iterations.

This notebook provides:
1. **Exploratory Data Analysis (EDA)** of the synthetic binary classification datasets.
2. **Agent API Guide**: Overview of the ADK submission structure (`agent.yaml`, `prompts/`, `configs/`).
3. **Dynamic AutoML Skill**: A complete, robust tabular AutoML baseline utilizing LightGBM, XGBoost, and CatBoost.
4. **Path Resolution Trick**: A workaround for resolving dataset path issues inside sandboxed skill executions.
5. **Packaging Instructions**: Step-by-step instructions on zipping and submitting your agent.

---

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

## 📊 1. Exploratory Data Analysis (EDA)

The competition provides 18 synthetic datasets simulating a binary classification task. Let's load the first dataset (`train_01`) and inspect its structure.

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/autonomous-agent-prediction-beta/data/train_01/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/autonomous-agent-prediction-beta/data/train_01/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
train_df.head()

### Target Class Distribution
Let's see if the classes are balanced.

In [ ]:
counts = train_df['target'].value_counts()
fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(x=counts.index, y=counts.values, palette="Blues_d", ax=ax)
for i, v in enumerate(counts.values):
    ax.text(i, v + 100, f"{v} ({v/len(train_df)*100:.1f}%)", ha="center", va="bottom", fontweight='bold')
ax.set_title("Target Column Distribution (train_01)")
ax.set_ylabel("Count")
plt.show()

### Categorical & Numerical Features
Let's inspect the data types of features.

In [ ]:
cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()
num_cols = train_df.select_dtypes(exclude=['object']).columns.tolist()
num_cols.remove('target')  # Target is integer but binary
if 'row_id' in num_cols:
    num_cols.remove('row_id')

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns (excluding ID/Target): {num_cols}")

## 🤖 2. Agent API Guide

During evaluation, your agent runs offline inside a sandbox container. The agent directory should be structured as follows:
```
submission/
├── agent.yaml           # YAML configuration defining model name and description
├── prompts/
│   └── system.md        # The system prompt instructions for the LLM agent
├── configs/
│   └── sampling.yaml    # Hyperparameters for model generation
└── skills/
    └── automl/          # Custom skills/scripts provided to the agent
        ├── SKILL.md     # Skill manifest file describing its usage
        └── scripts/
            └── run_automl.py   # Executable AutoML script
```

### Writable Directory Structure
By default, the agent operates in `/work` directory. It can interact with the environment via standard tool calls:
- `run_command`: Run terminal commands.
- `submit_predictions`: Submit a file to obtain leaderboard score.
- `select_submission`: Select final submissions for scoring.

Let's create the directories for our agent starter package.

In [ ]:
os.makedirs('submission/prompts', exist_ok=True)
os.makedirs('submission/configs', exist_ok=True)
os.makedirs('submission/skills/automl/scripts', exist_ok=True)
print("Created directories successfully!")

### Creating `agent.yaml`
This defines our agent configuration. We configure the agent to use `gemini-3.1-flash-lite` and grant it necessary tool permissions.

In [ ]:
%%writefile submission/agent.yaml
name: baseline_agent
type: LlmAgent
model: gemini-3.1-flash-lite
system_prompt: !include prompts/system.md
sampling_config: !include configs/sampling.yaml
allowed_tools:
  - run_command
  - submit_predictions
  - select_submission
  - get_status
  - list_skills
  - load_skill
  - run_skill_script

### Creating `prompts/system.md`
This serves as the system prompt (the instructions) for the model.

In [ ]:
%%writefile submission/prompts/system.md
You are an expert autonomous data scientist competing in a machine learning competition.

## Competition Task
{problem_description}

## Goal & Metric
Your objective is to maximize predictive performance on the test set. The metric is {metric_name}.

## Available Tools
- `list_skills`: Lists all available skills built-in to the workspace.
- `load_skill`: Loads details/instructions of a specific skill.
- `run_skill_script`: Runs an executable script bundled inside a skill.
- `run_command`: Run terminal commands.
- `submit_predictions`: Submit a file to obtain leaderboard score.
- `select_submission`: Select final submissions for scoring.

## Strategy
1. Start by listing the skills using `list_skills` to discover the automl tool.
2. Load the automl skill using `load_skill` and run its script `run_automl.py` via `run_skill_script`.
3. Inspect the validation scores printed in the console. If CatBoost CV AUC is high, train CatBoost with feature engineering enabled using the `--feature_engineering true` flag.
4. Submit your predictions and select the best submission ID for final scoring using `select_submission`.

### Creating `configs/sampling.yaml`
This defines the temperature and top-p sampling config for the LLM responses.

In [ ]:
%%writefile submission/configs/sampling.yaml
temperature: 0.1
top_p: 0.95

## 🛠️ 3. Dynamic AutoML Skill

Let's write a robust, modular, and parameterized AutoML script (`run_automl.py`) inside `skills/automl/scripts/`.

### The Path Resolution Workaround
When the harness executes `run_skill_script(skill_name="automl", file_path="scripts/run_automl.py")`, the sandbox container runs the script inside a **temporary directory** `/tmp/tmp...` instead of `/work`. Because the dataset files (`train.csv`, `test.csv`, etc.) reside in `/work`, a naive script will fail with `FileNotFoundError: train.csv`.

To solve this, we implement path resolution logic: the script will search parent directories, and if not found, it will look in `/work/` directly. This makes the skill completely robust!

In [ ]:
%%writefile submission/skills/automl/scripts/run_automl.py
#!/usr/bin/env python3
import os
import sys
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import warnings

# Suppress warnings for clean output
warnings.filterwarnings("ignore")

def resolve_path(path_str):
    """Find the path by searching parent directories and /work if not found locally."""
    path = Path(path_str)
    if path.exists():
        return str(path)
    # Search up to 4 parent directories
    for depth in range(1, 5):
        prefix = "../" * depth
        candidate = Path(prefix) / path_str
        if candidate.exists():
            return str(candidate)
    # Fallback to standard sandbox workspace root (/work)
    work_candidate = Path("/work") / path_str
    if work_candidate.exists():
        return str(work_candidate)
    return path_str

def resolve_output_path(output_path_str, resolved_train_path):
    """Save the output to the same parent directory as the training file if it is writable."""
    train_path = Path(resolved_train_path)
    if len(train_path.parts) > 1:
        parent_parts = train_path.parts[:-1]
        parent_dir = Path(*parent_parts)
        if os.access(parent_dir, os.W_OK):
            resolved_output = parent_dir / Path(output_path_str).name
            return str(resolved_output)
    return output_path_str

def optimize_blend_weights(oof_preds, y_train, models_to_run):
    """Find the best blending weights for the models using a grid search on OOF predictions."""
    import itertools
    best_score = -1
    best_weights = []
    
    n_models = len(models_to_run)
    if n_models == 1:
        return [1.0], roc_auc_score(y_train, oof_preds[models_to_run[0]])
        
    # Generate weight combinations that sum to 1.0 (with step size 0.1)
    combinations = []
    for weights in itertools.product(range(11), repeat=n_models):
        if sum(weights) == 10:
            combinations.append([w / 10.0 for w in weights])
            
    if not combinations:
        combinations = [[1.0 / n_models] * n_models]
        
    for weights in combinations:
        blend_oof = np.zeros(len(y_train))
        for idx, m in enumerate(models_to_run):
            blend_oof += oof_preds[m] * weights[idx]
            
        score = roc_auc_score(y_train, blend_oof)
        if score > best_score:
            best_score = score
            best_weights = weights
            
    return best_weights, best_score

def parse_args():
    parser = argparse.ArgumentParser(description="Automated Tabular Classifier")
    parser.add_argument("--train", type=str, default="train.csv", help="Path to train CSV")
    parser.add_argument("--test", type=str, default="test.csv", help="Path to test CSV")
    parser.add_argument("--sample_sub", type=str, default="sample_submission.csv", help="Path to sample submission CSV")
    parser.add_argument("--output", type=str, default="submission.csv", help="Path to save output submission")
    parser.add_argument("--feature_engineering", type=str, default="false", choices=["true", "false"], help="Enable feature engineering")
    parser.add_argument("--models", type=str, default="lgb,xgb,cb", help="Comma-separated list of models to train (lgb, xgb, cb)")
    return parser.parse_args()

def main():
    args = parse_args()
    fe_enabled = args.feature_engineering == "true"
    
    print(f"=== Starting AutoML ===")
    print(f"Loading data...")
    
    # Resolve paths dynamically
    resolved_train = resolve_path(args.train)
    resolved_test = resolve_path(args.test)
    resolved_sample_sub = resolve_path(args.sample_sub)
    resolved_output = resolve_output_path(args.output, resolved_train)
    
    train = pd.read_csv(resolved_train)
    test = pd.read_csv(resolved_test)
    sample_sub = pd.read_csv(resolved_sample_sub)
    
    # Identify target and ID columns
    target_col = None
    for col in train.columns:
        if col not in test.columns:
            target_col = col
            break
            
    if target_col is None:
        raise ValueError("Could not automatically identify the target column.")
        
    id_col = sample_sub.columns[0]
    print(f"Target column: {target_col}")
    print(f"ID column: {id_col}")
    print(f"Train shape: {train.shape}, Test shape: {test.shape}")
    
    # Separate features and target
    X_train = train.drop(columns=[id_col, target_col])
    y_train = train[target_col]
    X_test = test.drop(columns=[id_col])
    
    # Identify categorical, numeric, and ordinal columns
    cat_cols = []
    num_cols = []
    
    for col in X_train.columns:
        if X_train[col].dtype == "object" or X_train[col].dtype.name == "category":
            cat_cols.append(col)
        else:
            # If numerical but low cardinality (e.g. < 10 unique values), treat as categorical/ordinal
            if X_train[col].nunique() < 10:
                cat_cols.append(col)
            else:
                num_cols.append(col)
                
    print(f"Categorical features: {len(cat_cols)}")
    print(f"Numerical features: {len(num_cols)}")
    
    # Basic Imputation
    for col in X_train.columns:
        if col in cat_cols:
            fill_val = X_train[col].mode().iloc[0] if not X_train[col].mode().empty else "missing"
            X_train[col] = X_train[col].fillna(fill_val).astype(str)
            X_test[col] = X_test[col].fillna(fill_val).astype(str)
        else:
            fill_val = X_train[col].median()
            X_train[col] = X_train[col].fillna(fill_val)
            X_test[col] = X_test[col].fillna(fill_val)
            
    # Stratified K-Fold setup (needed early for OOF Target Encoding)
    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Feature Engineering (if enabled)
    if fe_enabled:
        print("Applying feature engineering...")
        # Row-wise statistics for numerical features
        if len(num_cols) > 0:
            for df in [X_train, X_test]:
                df["num_mean"] = df[num_cols].mean(axis=1)
                df["num_std"] = df[num_cols].std(axis=1)
                df["num_sum"] = df[num_cols].sum(axis=1)
                df["num_min"] = df[num_cols].min(axis=1)
                df["num_max"] = df[num_cols].max(axis=1)
            num_cols.extend(["num_mean", "num_std", "num_sum", "num_min", "num_max"])
            
        # OOF Target Encoding & Frequency Encoding for categoricals
        for col in cat_cols:
            # Frequency encoding
            freq = X_train[col].value_counts(normalize=True).to_dict()
            X_train[f"{col}_freq"] = X_train[col].map(freq)
            X_test[f"{col}_freq"] = X_test[col].map(freq).fillna(0)
            
            # Cross-validated Out-of-fold target encoding
            X_train[f"{col}_te"] = np.nan
            for train_idx, val_idx in skf.split(X_train, y_train):
                X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
                X_va = X_train.iloc[val_idx]
                
                means = y_tr.groupby(X_tr[col]).mean()
                X_train.iloc[val_idx, X_train.columns.get_loc(f"{col}_te")] = X_va[col].map(means)
                
            overall_means = y_train.groupby(X_train[col]).mean()
            X_test[f"{col}_te"] = X_test[col].map(overall_means)
            
            # Impute unmapped values with overall target mean
            global_mean = y_train.mean()
            X_train[f"{col}_te"] = X_train[f"{col}_te"].fillna(global_mean)
            X_test[f"{col}_te"] = X_test[f"{col}_te"].fillna(global_mean)
            
    # Categorical encoding (Ordinal Encoding)
    if len(cat_cols) > 0:
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
        X_test[cat_cols] = encoder.transform(X_test[cat_cols])
        
    print(f"Final training features shape: {X_train.shape}")
    
    models_to_run = args.models.split(",")
    oof_preds = {m: np.zeros(len(X_train)) for m in models_to_run}
    test_preds = {m: np.zeros(len(X_test)) for m in models_to_run}
    
    # Model configs
    model_factories = {
        "lgb": lambda: LGBMClassifier(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=6,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbose=-1,
            n_jobs=-1
        ),
        "xgb": lambda: XGBClassifier(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            eval_metric="logloss"
        ),
        "cb": lambda: CatBoostClassifier(
            iterations=1000,
            learning_rate=0.03,
            depth=6,
            random_seed=42,
            verbose=0,
            thread_count=-1
        )
    }
    
    for m in models_to_run:
        if m not in model_factories:
            print(f"Warning: Model {m} not supported. Skipping.")
            continue
            
        print(f"Training {m.upper()} classifier...")
        scores = []
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
            X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
            X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
            
            model = model_factories[m]()
            
            # Fit with early stopping if supported
            if m == "lgb":
                model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_va, y_va)],
                    callbacks=[]
                )
            elif m == "xgb":
                model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_va, y_va)],
                    verbose=False
                )
            elif m == "cb":
                model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_va, y_va)],
                    early_stopping_rounds=100,
                    verbose=False
                )
                
            val_pred = model.predict_proba(X_va)[:, 1]
            oof_preds[m][val_idx] = val_pred
            test_preds[m] += model.predict_proba(X_test)[:, 1] / n_splits
            
            fold_score = roc_auc_score(y_va, val_pred)
            scores.append(fold_score)
            
        oof_score = roc_auc_score(y_train, oof_preds[m])
        print(f"{m.upper()} 5-Fold CV Scores: {[round(s, 5) for s in scores]}")
        print(f"{m.upper()} Overall OOF AUC: {oof_score:.5f}")
        
    # Ensemble Blending
    if len(models_to_run) > 0:
        # Optimize weights using grid search on OOF predictions
        best_weights, ensemble_score = optimize_blend_weights(oof_preds, y_train, models_to_run)
        
        print(f"=== Optimized Ensemble Blending ===")
        print(f"Optimized weights: { {m: round(w, 2) for m, w in zip(models_to_run, best_weights)} }")
        print(f"Optimized Ensemble OOF AUC: {ensemble_score:.5f}")
        
        # Construct final OOF and test predictions
        final_oof = np.zeros(len(X_train))
        final_test = np.zeros(len(X_test))
        for idx, m in enumerate(models_to_run):
            final_oof += oof_preds[m] * best_weights[idx]
            final_test += test_preds[m] * best_weights[idx]
            
        # Save submission
        submission = pd.DataFrame({
            id_col: sample_sub[id_col],
            target_col: final_test
        })
        submission.to_csv(resolved_output, index=False)
        print(f"Saved submission to {resolved_output}")
        print(f"AutoML Completed Successfully!")
    else:
        print("Error: No models trained.")
        sys.exit(1)

if __name__ == "__main__":
    main()


### Creating the `SKILL.md` Manifest
This documents the skill description and exposes command parameters to our agent.

In [ ]:
%%writefile submission/skills/automl/SKILL.md
---
name: automl
description: Automated tabular binary classification skill utilizing LightGBM, XGBoost, and CatBoost models.
---

# AutoML Classifier Skill

Use this skill to automatically train models and generate submission predictions for any tabular binary classification dataset.

## How to use:
Run the AutoML script via `run_skill_script` inside the sandbox:
```python
run_skill_script(
    skill_name="automl",
    file_path="scripts/run_automl.py",
    args={"models": "cb", "feature_engineering": "true"}
)
```

## 🧪 4. Local Execution & Validation

Let's test our AutoML baseline script on the `train_01` dataset locally to verify it trains without path errors and produces a submission file.

In [ ]:
!python3 submission/skills/automl/scripts/run_automl.py --train /kaggle/input/competitions/autonomous-agent-prediction-beta/data/train_01/train.csv --test /kaggle/input/competitions/autonomous-agent-prediction-beta/data/train_01/test.csv --sample_sub /kaggle/input/competitions/autonomous-agent-prediction-beta/data/train_01/sample_submission.csv --output submission.csv --models cb --feature_engineering true

## 📦 5. Packaging & Submission

To submit your agent, navigate to the `submission` directory, zip the files so that `agent.yaml` is at the root, and upload the zip archive to Kaggle!

In [ ]:
!cd submission && zip -r ../submission.zip . && cd ..

### 🎉 Good Luck!
You now have a robust agent starter structure utilizing an AutoML baseline with path resolution fixes! Feel free to extend this by adding more skills, hyperparameter optimization, or custom models. If you found this notebook helpful, please leave an upvote! 👍